# **1. Processamento dos dados**

Este notebook transforma as tabelas coletadas em disciplinas únicas, prontas para a análise.

| camada | arquivos principais | papel |
|---|---|---|
| entrada | `data/raw/disciplines.csv`, `requisites.csv`, `curriculum.csv` | registros por código |
| saída | `data/processed/subjects.csv` | uma linha por disciplina única |
| auditoria | `data/processed/duplicates.csv`, `processing_audit.csv` | decisões e contagens |

A sequência é: normalizar textos → detectar equivalências → consolidar grupos → reescrever requisitos → exportar.

In [1]:
from __future__ import annotations

import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

### **1.1. Parâmetros e entradas**

`MIN_CHARS` controla o texto mínimo, `TEXT_DUPLICATE` define o limiar de equivalência por texto e `NAME_PERCENTILE` fixa o percentil que confirma uma equivalência por nome. As três tabelas de entrada são carregadas de `data/raw/`.

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DATA = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

MIN_CHARS = 80
TEXT_DUPLICATE = 0.90
NAME_PERCENTILE = 99

disciplines = pd.read_csv(RAW_DATA / "disciplines.csv", dtype=str).fillna("")
requisites = pd.read_csv(RAW_DATA / "requisites.csv", dtype={"course_code": str})
curriculum = pd.read_csv(RAW_DATA / "curriculum.csv")
print(f"{len(disciplines)} disciplinas, {len(requisites)} linhas de requisito, {len(curriculum)} linhas de grade")

636 disciplinas, 1292 linhas de requisito, 1089 linhas de grade


## **2. Texto**

Os campos passam para Unicode NFC e espaços repetidos são removidos. O corpo analisado é

$$
texto = ementa + objetivos + programa.
$$

Registros desativados ou com menos de 80 caracteres permanecem em `data/processed/disciplines.csv`, mas não entram na análise.

A sigla é o prefixo alfabético do código (SCC, SMA, SEL...). Códigos numéricos não têm sigla departamental e recebem a da unidade indicada pelos dois primeiros dígitos.

In [3]:
TEXT_COLUMNS = ["name", "name_en", "ementa", "objectives", "program", "teachers"]
for column in TEXT_COLUMNS:
    disciplines[column] = disciplines[column].map(lambda v: " ".join(unicodedata.normalize("NFC", v).split()))

disciplines["body"] = (disciplines["ementa"] + " " + disciplines["objectives"] + " " + disciplines["program"]).str.strip()
disciplines["chars"] = disciplines["body"].str.len()
NUMERIC_UNITS = {"18": "EESC", "55": "ICMC", "75": "IQSC", "76": "IFSC", "90": "LCE", "97": "EESC-ICMC"}
disciplines["prefix"] = disciplines["code"].str.extract(r"^([A-Z]+)")[0].fillna(disciplines["code"].str[:2].map(NUMERIC_UNITS))
disciplines["status"] = np.select(
    [disciplines["deactivation"] != "", disciplines["chars"] < MIN_CHARS],
    ["desativada", "sem texto suficiente"], "analisada")

assert disciplines["prefix"].notna().all(), "código numérico sem unidade mapeada"
display(disciplines["status"].value_counts().rename("disciplinas").to_frame().T)
disciplines.loc[disciplines["status"] != "analisada", ["code", "name", "status", "chars"]]

status,analisada,sem texto suficiente
disciplinas,632,4


,code,name,status,chars
374,SMA0406,,sem texto suficiente,0
614,SSC0112,,sem texto suficiente,0
615,SSC0114,,sem texto suficiente,0
616,SSC0143,,sem texto suficiente,0


## **3. Duplicatas**

Duas disciplinas são consideradas a mesma quando:

1. **têm o mesmo nome normalizado e textos compatíveis**: o nome é comparado sem acentos, maiúsculas, artigos e preposições, com plural simplificado e sem anotações entre parênteses como “(semipresencial)”. A numeração (I, II, III, também escrita como 1, 2, 3) é mantida, para que Cálculo I e Cálculo II continuem distintos. O nome apenas indica a candidata: o par só é unido se o cosseno TF-IDF superar o percentil 99 de todos os pares. Assim, homônimas com conteúdos distintos, como Física III do IFSC (eletromagnetismo) e Física 3 da Licenciatura em Ciências Exatas (termodinâmica), permanecem separadas, ou
2. **têm nomes diferentes, mas texto quase idêntico**: cosseno TF-IDF ≥ 0,90, com a mesma numeração.

$$
\operatorname{sim}(a,b)=\frac{\mathbf{x}_a\cdot\mathbf{x}_b}{\lVert\mathbf{x}_a\rVert\,\lVert\mathbf{x}_b\rVert}\geq 0{,}90.
$$

Os grupos são fechados por união-busca: se A equivale a B e B equivale a C, os três códigos formam uma disciplina única.

In [4]:
STOP = {"a", "o", "as", "os", "de", "da", "do", "das", "dos", "e", "em", "para", "com", "ao", "aos", "na", "no", "nas", "nos"}
ROMAN = re.compile(r"^(i|ii|iii|iv|v|vi|vii|viii|ix|x|\d+)$")


def fold(value: str) -> str:
    value = unicodedata.normalize("NFKD", value)
    return "".join(c for c in value if not unicodedata.combining(c)).lower()


ARABIC = {"1": "i", "2": "ii", "3": "iii", "4": "iv", "5": "v", "6": "vi"}


def name_tokens(name: str) -> list[str]:
    tokens = re.findall(r"[a-z0-9]+", re.sub(r"\(.*?\)", " ", fold(name)))
    return [ARABIC.get(t, t) for t in tokens]


def name_key(name: str) -> str:
    out = []
    for token in name_tokens(name):
        if token in STOP:
            continue
        if not ROMAN.match(token) and len(token) > 3 and token.endswith("s"):
            token = token[:-1]
        out.append(token)
    return " ".join(out)


def numbering(name: str) -> tuple[str, ...]:
    return tuple(t for t in name_tokens(name) if ROMAN.match(t))


df = disciplines[disciplines["status"] == "analisada"].reset_index(drop=True)
df["name_key"] = df["name"].map(name_key)
df["numbering"] = df["name"].map(numbering)

tfidf = TfidfVectorizer(strip_accents="unicode", ngram_range=(1, 2), sublinear_tf=True)
X = tfidf.fit_transform(df["body"])
S = (X @ X.T).toarray()
np.fill_diagonal(S, 0)
NAME_CONFIRM = float(np.percentile(S[np.triu_indices(len(df), 1)], NAME_PERCENTILE))

parent = list(range(len(df)))
def root(i: int) -> int:
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

links, rejected = [], []
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        same_name = df.at[i, "name_key"] == df.at[j, "name_key"]
        by_name = same_name and S[i, j] >= NAME_CONFIRM
        by_text = S[i, j] >= TEXT_DUPLICATE and df.at[i, "numbering"] == df.at[j, "numbering"]
        if same_name and not by_name:
            rejected.append({"code_a": df.at[i, "code"], "name_a": df.at[i, "name"], "code_b": df.at[j, "code"],
                             "name_b": df.at[j, "name"], "text_similarity": round(float(S[i, j]), 3)})
        if by_name or by_text:
            parent[root(j)] = root(i)
            links.append({"code_a": df.at[i, "code"], "name_a": df.at[i, "name"], "code_b": df.at[j, "code"],
                          "name_b": df.at[j, "name"], "rule": "mesmo nome" if by_name else "mesmo texto",
                          "text_similarity": round(float(S[i, j]), 3)})
df["group"] = [root(i) for i in range(len(df))]
links, rejected = pd.DataFrame(links), pd.DataFrame(rejected)
print(f"{len(df)} códigos → {df['group'].nunique()} disciplinas únicas · confirmação por nome: cosseno ≥ {NAME_CONFIRM:.3f}")
display(links["rule"].value_counts().rename("pares").to_frame().T)
display(rejected.sort_values("text_similarity"))
links[links["rule"] == "mesmo texto"]

632 códigos → 528 disciplinas únicas · confirmação por nome: cosseno ≥ 0.099


rule,mesmo nome,mesmo texto
pares,132,9


,code_a,name_a,code_b,name_b,text_similarity
1,7600107,Física III,SLC0628,Física 3,0.023
2,SEL0624,Projeto de Formatura I,SSC0670,Projeto de Formatura I,0.044
4,SMA0393,Cálculo III (semipresencial),SMA0803,Cálculo III,0.050
3,SMA0355,Cálculo III,SMA0803,Cálculo III,0.057
0,1800122,Estágio Supervisionado,SME0284,Estágio Supervisionado,0.071


,code_a,name_a,code_b,name_b,rule,text_similarity
0,1800124,Atividades de Extensão para Engenharia I,SLC0700,Atividades de Extensão para Licenciatura 1,mesmo texto,1.000
1,1800125,Atividades de Extensão para Engenharia II,SLC0701,Atividades de Extensão para Licenciatura 2,mesmo texto,1.000
2,5500003,Robótica Educacional,SLC0616,Robótica Educacional aplicada à Matemática,mesmo texto,1.000
14,SCC0209,Empreendedores em Informática,SSC0570,Empreendedorismo,mesmo texto,0.941
57,SEL0607,Fundamentos de Semicondutores,SEL0450,Semicondutores,mesmo texto,0.937
128,SSC0524,"Verificação, Validação e Teste de Software",SSC0959,Teste e Inspeção de Software,mesmo texto,1.000
135,SSC0593,Projeto de Graduação I,SSC0670,Projeto de Formatura I,mesmo texto,0.935
139,SEM0388,Princípios de Metrologia Industrial,SEM0566,Sistemas e Métodos de Controle de Processos,mesmo texto,1.000
140,SEL0454,Introdução aos Sistemas Digitais,SEL0455,Laboratório de Sistemas Digitais,mesmo texto,1.000


### **3.1. Grupos encontrados**

A tabela destaca os grupos com mais códigos e permite conferir se nomes diferentes foram consolidados de forma coerente.

In [5]:
sizes = df.groupby("group").agg(codes=("code", list), names=("name", lambda v: sorted(set(v))), n=("code", "size"))
sizes.sort_values("n", ascending=False).head(12)

,codes,names,n
group,,,
227,"[SLC0607, SMA0301, SMA0353, SMA0392, SMA0501, ...","[Cálculo 1, Cálculo I, Cálculo I (semipresenci...",6
228,"[SLC0608, SMA0332, SMA0354, SMA0390, SMA0802]","[Cálculo II, Cálculo II (semipresencial)]",5
61,"[SCC0222, SSC0502, SSC0601, SSC0801]",[Laboratório de Introdução à Ciência de Comput...,4
60,"[SCC0221, SSC0501, SSC0600, SSC0800]",[Introdução à Ciência de Computação I],4
383,"[SME0104, SME0500, SME0602, SME0300]",[Cálculo Numérico],4
47,"[SCC0204, SCC0504, SCC0604, SSC0103]",[Programação Orientada a Objetos],4
192,"[SEL0628, SSC0118, SEL0414]",[Sistemas Digitais],3
68,"[SCC0230, SCC0530, SCC0630]",[Inteligência Artificial],3
71,"[SCC0240, SCC0540, SCC0640]",[Bases de Dados],3


## **4. Disciplinas únicas**

Cada grupo vira uma linha em `data/processed/subjects.csv`. O código representante aparece no maior número de cursos. Em empate, vence o registro com mais texto. A sigla da disciplina única é a do representante.

| prioridade do semestre ideal | fonte |
|---:|---|
| 1 | disciplinas obrigatórias nas grades selecionadas |
| 2 | qualquer ocorrência nas grades selecionadas |
| 3 | listas de requisitos dos demais cursos |

Entre cursos usa-se a moda e, em empate, o semestre mais cedo.

In [6]:
course_sets = pd.concat([
    requisites[["code", "course_code"]].rename(columns={"course_code": "course"}),
    curriculum.assign(course=curriculum["course_id"].str.split("-").str[0])[["code", "course"]],
]).drop_duplicates()
courses_by_code = course_sets.groupby("code")["course"].apply(set).to_dict()
grade_periods = curriculum.assign(course=curriculum["course_id"])[["code", "course", "semester", "kind"]].dropna(subset=["semester"])
req_periods = requisites[["code", "course_code", "ideal_period"]].rename(columns={"course_code": "course", "ideal_period": "semester"}).dropna()
SOURCES = [grade_periods[grade_periods["kind"] == "obrigatória"], grade_periods, req_periods]


def modal(values) -> int | None:
    counts = Counter(int(v) for v in values)
    return min(counts, key=lambda v: (-counts[v], v)) if counts else None


def ideal_period(codes: list[str]) -> int | None:
    for source in SOURCES:
        rows = source[source["code"].isin(codes)]
        if len(rows):
            return modal(rows.groupby("course")["semester"].min())
    return None

as_int = lambda v: int(float(v)) if v else 0


def display_name(names: pd.Series) -> str:
    """Nome mais frequente entre as versões, sem anotações como “(semipresencial)”."""
    cleaned = names.map(lambda v: re.sub(r"\s*\(.*?\)", "", v).strip())
    return Counter(cleaned).most_common(1)[0][0]


subject_rows = []
for group, members in df.groupby("group"):
    members = members.assign(n_courses=members["code"].map(lambda c: len(courses_by_code.get(c, ()))))
    rep = members.sort_values(["n_courses", "chars"], ascending=False).iloc[0]
    courses = set().union(*(courses_by_code.get(c, set()) for c in members["code"]))
    teachers = sorted({t.strip() for v in members["teachers"] for t in v.split(";") if t.strip()})
    subject_rows.append({
        "subject_id": rep["code"], "name": display_name(members["name"]), "codes": " ".join(sorted(members["code"])), "n_codes": len(members),
        "prefix": rep["prefix"], "prefixes": " ".join(sorted(set(members["prefix"]))), "department": rep["department"],
        "credits": as_int(rep["credits_class"]) + as_int(rep["credits_work"]) or None,
        "workload_hours": as_int(rep["workload_hours"]) or None,
        "ideal_period": ideal_period(list(members["code"])), "n_courses": len(courses),
        "teachers": "; ".join(teachers), "ementa": rep["ementa"], "objectives": rep["objectives"], "program": rep["program"],
        "body_all": " ".join(dict.fromkeys(members.sort_values("chars", ascending=False)["body"])),
    })
subjects = pd.DataFrame(subject_rows).sort_values("subject_id").reset_index(drop=True)
subject_of = {code: subjects.at[k, "subject_id"] for k, row in subjects.iterrows() for code in row["codes"].split()}
disciplines["subject_id"] = disciplines["code"].map(subject_of)
subjects[["subject_id", "name", "codes", "n_courses", "ideal_period"]].sort_values("n_courses", ascending=False).head(10)

,subject_id,name,codes,n_courses,ideal_period
355,SMA0393,Cálculo III,SMA0355 SMA0393,22,3.0
354,SMA0392,Cálculo I,SLC0607 SMA0301 SMA0353 SMA0392 SMA0501 SMA0801,19,1.0
332,SMA0354,Cálculo II,SLC0608 SMA0332 SMA0354 SMA0390 SMA0802,16,2.0
356,SMA0394,Geometria Analítica,SMA0300 SMA0394 SMA0800,14,1.0
400,SME0300,Cálculo Numérico,SME0104 SME0300 SME0500 SME0602,10,3.0
81,SCC0240,Bases de Dados,SCC0240 SCC0540 SCC0640,9,5.0
403,SME0320,Estatística I,SME0320 SME0620,9,5.0
97,SCC0276,Aprendizado de Máquina,SCC0276 SME0829,8,5.0
368,SME0130,Redes Complexas,SME0130,8,7.0
93,SCC0270,Redes Neurais e Aprendizado Profundo,SCC0270,8,5.0


## **5. Requisitos por disciplina única**

Cada código é substituído por sua disciplina única. Um par requisito → disciplina acumula o número de cursos e informa se existe ao menos uma relação forte. Pares internos ao mesmo grupo e códigos fora da análise são descartados. A saída é `data/processed/subject_requisites.csv`.

In [7]:
req = requisites.assign(source=requisites["requisite"].map(subject_of), target=requisites["code"].map(subject_of))
dropped = req["source"].isna() | req["target"].isna() | (req["source"] == req["target"])
subject_requisites = (req[~dropped].assign(strong=lambda t: t["relation"].eq("requisito"))
                      .groupby(["source", "target"])
                      .agg(n_courses=("course_code", "nunique"), strong=("strong", "any"),
                           relations=("relation", lambda v: "; ".join(sorted(set(v)))))
                      .reset_index())
print(f"{len(requisites)} linhas → {len(subject_requisites)} pares entre disciplinas únicas ({int(dropped.sum())} linhas descartadas)")
subject_requisites.sort_values("n_courses", ascending=False).head(8)

1292 linhas → 702 pares entre disciplinas únicas (60 linhas descartadas)


,source,target,n_courses,strong,relations
474,SMA0392,SMA0354,16,True,requisito; requisito fraco
426,SMA0354,SMA0393,12,True,requisito; requisito fraco
627,SME0822,SME0878,8,True,requisito; requisito fraco
161,SCC0240,SCC0244,7,False,requisito fraco
148,SCC0230,SCC0270,7,False,indicação de conjunto; requisito fraco
147,SCC0230,SCC0244,7,False,requisito fraco
483,SMA0392,SME0320,6,False,requisito fraco
472,SMA0392,SEM0576,6,False,requisito fraco


## **6. Exportação e auditoria**

| arquivo em `data/processed/` | conteúdo |
|---|---|
| `disciplines.csv` | registros originais com estado e grupo |
| `subjects.csv` | disciplinas únicas |
| `duplicates.csv` | pares unidos e regra aplicada |
| `subject_requisites.csv` | requisitos consolidados |
| `processing_audit.csv` | contagens da etapa |

In [8]:
outputs = {
    "disciplines.csv": disciplines.drop(columns=["chars"]),
    "subjects.csv": subjects,
    "duplicates.csv": links,
    "subject_requisites.csv": subject_requisites,
}
audit = pd.DataFrame([
    {"metric": "disciplines_collected", "value": len(disciplines)},
    {"metric": "disciplines_analyzed", "value": int((disciplines["status"] == "analisada").sum())},
    {"metric": "excluded_no_text", "value": int((disciplines["status"] == "sem texto suficiente").sum())},
    {"metric": "excluded_deactivated", "value": int((disciplines["status"] == "desativada").sum())},
    {"metric": "duplicate_pairs_same_name", "value": int((links["rule"] == "mesmo nome").sum())},
    {"metric": "duplicate_pairs_same_text", "value": int((links["rule"] == "mesmo texto").sum())},
    {"metric": "same_name_rejected", "value": len(rejected)},
    {"metric": "name_confirm_threshold", "value": f"{NAME_CONFIRM:.3f}"},
    {"metric": "unique_subjects", "value": len(subjects)},
    {"metric": "subjects_with_several_codes", "value": int((subjects["n_codes"] > 1).sum())},
    {"metric": "max_codes_per_subject", "value": int(subjects["n_codes"].max())},
    {"metric": "subject_requisite_pairs", "value": len(subject_requisites)},
])
outputs["processing_audit.csv"] = audit
for name, frame in outputs.items():
    frame.to_csv(PROCESSED / name, index=False)
display(audit)
pd.DataFrame([{"artifact": name, "rows": len(frame), "directory": str(PROCESSED.relative_to(ROOT))} for name, frame in outputs.items()])

,metric,value
0,disciplines_collected,636
1,disciplines_analyzed,632
2,excluded_no_text,4
3,excluded_deactivated,0
4,duplicate_pairs_same_name,132
5,duplicate_pairs_same_text,9
6,same_name_rejected,5
7,name_confirm_threshold,0.099
8,unique_subjects,528
9,subjects_with_several_codes,78


,artifact,rows,directory
0,disciplines.csv,636,data/processed
1,subjects.csv,528,data/processed
2,duplicates.csv,141,data/processed
3,subject_requisites.csv,702,data/processed
4,processing_audit.csv,12,data/processed
